# Intent Classification Lab: From SVM Baseline to LoRA Fine-Tuning

In this lab, we work on the **Out-of-Scope Intent Classification Dataset (CLINC/OOS)**.
The input is a short user query, and the output is an intent label. We focus on the in-scope intent classification setting first, and we keep the out-of-scope data available for later analysis.

We will:

1. Load the raw JSON files from `./raw/`.
2. Build a local train/validation split from the official train split.
3. Build a classical **SVM + TF-IDF** baseline.
4. Add synthetic training data.
5. Fine-tune a small language model with **LoRA** using Unsloth.
6. Compare all models on the same local validation set.

Dataset source: Larson et al., *An Evaluation Dataset for Intent Classification and Out-of-Scope Prediction* (EMNLP 2019).


## 0. Runtime Setup

This notebook assumes that the raw CLINC/OOS JSON files are already stored in `./raw/`.
A simple directory layout is:

- `./raw/is_train.json`
- `./raw/is_val.json`
- `./raw/is_test.json`
- `./raw/oos_train.json`
- `./raw/oos_val.json`
- `./raw/oos_test.json`
- `./data/` for locally prepared files

If you are using Colab, install the required packages before running the rest of the notebook.


In [1]:
# Uncomment this cell in Colab if the packages are not installed yet.
# !uv pip install unsloth vllm --torch-backend=auto 
# !uv pip install notebook scikit-learn

## 1. Imports

We import `unsloth` before `trl`, `transformers`, and `peft`-related training utilities so that all speed and memory optimizations are applied correctly.


In [2]:
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List
import json

import pandas as pd
import torch
from datasets import Dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from unsloth import FastLanguageModel
from transformers import TrainingArguments
from trl import SFTTrainer


/home/dryjins/miniconda3/envs/lora/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
WARNING 05-12 16:25:09 [interface.py:525] Using 'pin_memory=False' as WSL is detected. This may slow down the performance.
🦥 Unsloth Zoo will now patch everything to make training faster!


## 2. Paths and Raw Data

This section reads the original CLINC/OOS JSON files from `./raw/`.
The in-scope files contain the 150 target intents. The out-of-scope files contain the special `oos` label.

For the main lab, we use the in-scope training data to build the local split. The OOS files are loaded as optional analysis data.


In [3]:
RAW_DIR = Path('./raw')
DATA_DIR = Path('./data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

IN_SCOPE_FILES = {
    'train': RAW_DIR / 'is_train.json',
    'val': RAW_DIR / 'is_val.json',
    'test': RAW_DIR / 'is_test.json',
}
OOS_FILES = {
    'train': RAW_DIR / 'oos_train.json',
    'val': RAW_DIR / 'oos_val.json',
    'test': RAW_DIR / 'oos_test.json',
}

for split_name, path in {**IN_SCOPE_FILES, **{f'oos_{k}': v for k, v in OOS_FILES.items()}}.items():
    print(split_name, path, path.exists())


train raw/is_train.json True
val raw/is_val.json True
test raw/is_test.json True
oos_train raw/oos_train.json True
oos_val raw/oos_val.json True
oos_test raw/oos_test.json True


The CLINC/OOS JSON files are usually simple lists of examples. Different mirrors may store each example as a string, a list, or a small dict, so the loader below is defensive by design.


In [4]:
def _normalize_example(example, default_label=None):
    """Normalize one raw CLINC/OOS example into a dict with text and intent."""
    if isinstance(example, dict):
        text = example.get('text') or example.get('utterance') or example.get('query') or example.get('sentence')
        label = example.get('intent') or example.get('label') or example.get('intent_name') or default_label
        return {'text': text, 'intent': label}

    if isinstance(example, list):
        if len(example) == 2:
            return {'text': example[0], 'intent': example[1] if default_label is None else default_label}
        if len(example) == 1:
            return {'text': example[0], 'intent': default_label}

    if isinstance(example, str):
        return {'text': example, 'intent': default_label}

    raise ValueError(f'Unsupported example format: {type(example)}')


def load_clinc_json(path: Path, default_label=None) -> pd.DataFrame:
    """Load a CLINC/OOS JSON split into a normalized pandas DataFrame."""
    with path.open('r', encoding='utf-8') as f:
        payload = json.load(f)

    if isinstance(payload, dict):
        for candidate_key in ['data', 'examples', 'records']:
            if candidate_key in payload:
                payload = payload[candidate_key]
                break

    if not isinstance(payload, list):
        raise ValueError(f'Expected a list-like payload in {path}, got {type(payload)}')

    rows = [_normalize_example(example, default_label=default_label) for example in payload]
    df = pd.DataFrame(rows)
    df = df.dropna(subset=['text', 'intent']).reset_index(drop=True)
    return df


In [5]:
is_train_df = load_clinc_json(IN_SCOPE_FILES['train'])
is_val_df = load_clinc_json(IN_SCOPE_FILES['val'])
is_test_df = load_clinc_json(IN_SCOPE_FILES['test'])

oos_train_df = load_clinc_json(OOS_FILES['train'], default_label='oos')
oos_val_df = load_clinc_json(OOS_FILES['val'], default_label='oos')
oos_test_df = load_clinc_json(OOS_FILES['test'], default_label='oos')

print('In-scope train shape:', is_train_df.shape)
print('In-scope val shape:', is_val_df.shape)
print('In-scope test shape:', is_test_df.shape)
print('OOS train shape:', oos_train_df.shape)
print(is_train_df.head())
print(is_train_df['intent'].nunique())


In-scope train shape: (15000, 2)
In-scope val shape: (3000, 2)
In-scope test shape: (4500, 2)
OOS train shape: (100, 2)
                                                text     intent
0  what expression would i use to say i love you ...  translate
1  can you tell me how to say 'i do not speak muc...  translate
2  what is the equivalent of, 'life is good' in f...  translate
3  tell me how to say, 'it is a beautiful morning...  translate
4  if i were mongolian, how would i say that i am...  translate
150


## 3. Create the Main Training Frame

For the core lab, we focus on in-scope intent classification.
We combine the official in-scope train and validation splits, then create our own local validation split for fair model comparison inside the classroom.

The official OOS files stay available for optional extension experiments.


In [6]:
raw_train_df = pd.concat([is_train_df, is_val_df], ignore_index=True)
raw_train_df = raw_train_df[['text', 'intent']].copy()
raw_train_df['id'] = raw_train_df.index.astype(int)

print('Combined in-scope training pool:', raw_train_df.shape)
print('Number of intents:', raw_train_df['intent'].nunique())
print(raw_train_df['intent'].value_counts().head(10))


Combined in-scope training pool: (18000, 3)
Number of intents: 150
intent
translate               120
transfer                120
timer                   120
definition              120
meaning_of_life         120
insurance_change        120
find_phone              120
travel_alert            120
pto_request             120
improve_credit_score    120
Name: count, dtype: int64


## 4. Create a Local Validation Split

The local validation split lets us compare the SVM baseline and the LoRA model on exactly the same examples.
We stratify by intent so that the class distribution is preserved as much as possible.


In [7]:
train_base_df, test_local_df = train_test_split(
    raw_train_df[['id', 'text', 'intent']],
    test_size=0.1,
    random_state=42,
    stratify=raw_train_df['intent'],
)

train_base_df = train_base_df.reset_index(drop=True)
test_local_df = test_local_df.reset_index(drop=True)

train_base_path = DATA_DIR / 'train_base.csv'
test_local_path = DATA_DIR / 'test_local.csv'

train_base_df.to_csv(train_base_path, index=False)
test_local_df.to_csv(test_local_path, index=False)

print(train_base_df.shape, test_local_df.shape)
print(train_base_df.head())


(16200, 3) (1800, 3)
      id                                            text            intent
0   4806                          what is my health plan         insurance
1  16669             schedule my meeting with jim at 3pm  schedule_meeting
2  17294        do you need shots before going to russia          vaccines
3   9110                what's my current spending limit      credit_limit
4   5202  what's the amount of air in my tires right now     tire_pressure


## 5. SVM Baseline

A TF-IDF + Linear SVM baseline is strong, cheap, and easy to interpret.
It gives us a useful classical reference point before we move to LoRA fine-tuning.


In [8]:
svm_pipeline = Pipeline(
    steps=[
        ('tfidf', TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, max_features=50000)),
        ('clf', LinearSVC()),
    ]
)

svm_pipeline.fit(train_base_df['text'], train_base_df['intent'])
svm_pred = svm_pipeline.predict(test_local_df['text'])

svm_acc = accuracy_score(test_local_df['intent'], svm_pred)
svm_macro_f1 = f1_score(test_local_df['intent'], svm_pred, average='macro')

print('SVM accuracy:', round(svm_acc, 4))
print('SVM macro F1:', round(svm_macro_f1, 4))


SVM accuracy: 0.95
SVM macro F1: 0.9499


## 6. Optional: Add Synthetic Data

This lab keeps the synthetic-data step optional.
If you generate synthetic examples, they should follow the same schema as `train_base.csv`: `text`, `intent`.


In [9]:
synthetic_path = DATA_DIR / 'train_synth.csv'

if synthetic_path.exists():
    synthetic_df = pd.read_csv(synthetic_path)
    required_columns = {'text', 'intent'}
    if not required_columns.issubset(synthetic_df.columns):
        raise ValueError(f'Synthetic data must contain {sorted(required_columns)}')

    train_aug_df = pd.concat(
        [train_base_df[['text', 'intent']], synthetic_df[['text', 'intent']]],
        ignore_index=True,
    )
    print('Using synthetic data.')
else:
    synthetic_df = pd.DataFrame(columns=['text', 'intent'])
    train_aug_df = train_base_df[['text', 'intent']].copy()
    print('No synthetic data file found. Using only base training data.')

print(train_aug_df.shape)


No synthetic data file found. Using only base training data.
(16200, 2)


## 7. Label Space

We build a fixed label vocabulary from the in-scope training pool.
This keeps the prompt format and evaluation stable across all models.


In [10]:
label_list = sorted(train_aug_df['intent'].unique().tolist())
label_to_id = {label: idx for idx, label in enumerate(label_list)}
id_to_label = {idx: label for label, idx in label_to_id.items()}

print('Number of labels:', len(label_list))
print(label_list[:20])


Number of labels: 150
['accept_reservations', 'account_blocked', 'alarm', 'application_status', 'apr', 'are_you_a_bot', 'balance', 'bill_balance', 'bill_due', 'book_flight', 'book_hotel', 'calculator', 'calendar', 'calendar_update', 'calories', 'cancel', 'cancel_reservation', 'car_rental', 'card_declined', 'carry_on']


## 8. Prompt Formatting for LoRA Training

We cast intent classification as a compact instruction-following problem.
The prompt is intentionally simple so that students can focus on the adaptation pipeline rather than prompt engineering tricks.


In [11]:
def build_instruction(text: str, labels: List[str]) -> str:
    """Build a short instruction prompt for intent classification."""
    label_str = ', '.join(labels)
    return (
        'You are an intent classification model.\n'
        f'Possible labels: {label_str}\n\n'
        f'Input: {text}\n'
        'Output:'
    )


def format_training_example(row: pd.Series, labels: List[str]) -> Dict[str, str]:
    """Format one supervised training example for SFT."""
    prompt = build_instruction(row['text'], labels)
    full_text = f"{prompt} {row['intent']}"
    return {'text': full_text}


train_sft_df = train_aug_df.copy()
train_sft_records = [format_training_example(row, label_list) for _, row in train_sft_df.iterrows()]
train_sft_dataset = Dataset.from_pandas(pd.DataFrame(train_sft_records))

print(train_sft_dataset[0]['text'])


You are an intent classification model.
Possible labels: accept_reservations, account_blocked, alarm, application_status, apr, are_you_a_bot, balance, bill_balance, bill_due, book_flight, book_hotel, calculator, calendar, calendar_update, calories, cancel, cancel_reservation, car_rental, card_declined, carry_on, change_accent, change_ai_name, change_language, change_speed, change_user_name, change_volume, confirm_reservation, cook_time, credit_limit, credit_limit_change, credit_score, current_location, damaged_card, date, definition, direct_deposit, directions, distance, do_you_have_pets, exchange_rate, expiration_date, find_phone, flight_status, flip_coin, food_last, freeze_account, fun_fact, gas, gas_type, goodbye, greeting, how_busy, how_old_are_you, improve_credit_score, income, ingredient_substitution, ingredients_list, insurance, insurance_change, interest_rate, international_fees, international_visa, jump_start, last_maintenance, lost_luggage, make_call, maybe, meal_suggestion, 

## 9. Unsloth Configuration

This configuration is intentionally conservative for classroom use.
The target model is `LiquidAI/LFM2.5-1.2B-Instruct`, and the LoRA setup is designed to be easy to modify during live teaching.


In [12]:
@dataclass
class LabConfig:
    """
    Configuration values for the classroom LoRA fine-tuning lab.
    Tuned for a 6GB RTX 3060 with LFM2.5-1.2B + 4-bit LoRA.
    """
    model_name: str = "LiquidAI/LFM2.5-1.2B-Instruct"
    max_seq_length: int = 256          # was 512
    load_in_4bit: bool = True
    r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.0
    learning_rate: float = 2e-4

    per_device_train_batch_size: int = 1   # was 4
    gradient_accumulation_steps: int = 8   # was 4 (keeps effective batch ~8)
    warmup_steps: int = 5
    max_steps: int = 40                    # was 60 (조금 줄여서 안정성 확보)
    logging_steps: int = 5
    output_dir: str = "outputs/lora_clinc"


config = LabConfig()
config


LabConfig(model_name='LiquidAI/LFM2.5-1.2B-Instruct', max_seq_length=256, load_in_4bit=True, r=16, lora_alpha=32, lora_dropout=0.0, learning_rate=0.0002, per_device_train_batch_size=1, gradient_accumulation_steps=8, warmup_steps=5, max_steps=40, logging_steps=5, output_dir='outputs/lora_clinc')

## 10. Load the Model and Add LoRA Adapters

Some architectures use slightly different module names, so the `target_modules` list may need a quick check before class.
The defaults below are a strong starting point for many decoder-only instruction models.


In [13]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=config.model_name,
    max_seq_length=config.max_seq_length,
    dtype=None,
    load_in_4bit=config.load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=config.r,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

print(tokenizer.pad_token, tokenizer.eos_token)


==((====))==  Unsloth 2026.5.2: Fast Lfm2 patching. Transformers: 4.57.6. vLLM: 0.19.1.
   \\   /|    NVIDIA GeForce RTX 3060 Laptop GPU. Num GPUs = 1. Max memory: 6.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
<|pad|> <|im_end|>


## 11. Trainer Setup

We use `SFTTrainer` for simplicity.
This is not the only way to fine-tune an intent classifier, but it is enough to expose the core LoRA workflow in one lab session.


In [14]:
training_args = TrainingArguments(
    output_dir=config.output_dir,
    per_device_train_batch_size=config.per_device_train_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    warmup_steps=config.warmup_steps,
    max_steps=config.max_steps,
    learning_rate=config.learning_rate,
    logging_steps=config.logging_steps,
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='linear',
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    seed=42,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_sft_dataset,
    dataset_text_field='text',
    max_seq_length=config.max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

trainer


Unsloth: Tokenizing ["text"] (num_proc=12): 100%|█████████████████████████████████████| 16200/16200 [00:06<00:00, 2358.84 examples/s]


## 12. Train

Run this cell only in a compatible GPU environment.
If classroom time is tight, you can reduce `max_steps` further.


In [15]:
trainer_stats = trainer.train()
trainer_stats

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 16,200 | Num Epochs = 1 | Total steps = 40
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 884,736 of 1,171,225,344 (0.08% trained)


Step,Training Loss
5,3.108600
10,2.659900
15,2.325600
20,2.046300
25,1.778200
30,1.545900
35,1.380900
40,1.272500


TrainOutput(global_step=40, training_loss=2.01474871635437, metrics={'train_runtime': 97.2705, 'train_samples_per_second': 3.29, 'train_steps_per_second': 0.411, 'total_flos': 1214325548259840.0, 'train_loss': 2.01474871635437, 'epoch': 0.019753086419753086})

## 13. Local Evaluation for the LoRA Model

We generate one label string per input and map it back into the closed label set.
For a classroom baseline, exact-string postprocessing is usually sufficient.


In [18]:
FastLanguageModel.for_inference(model)

def batch_predict_intent(text_list: List[str], labels: List[str]) -> List[str]:
    """Generate one intent label per text for a batch of examples."""
    prompts = [build_instruction(text, labels) for text in text_list]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=8, use_cache=True)
    decoded_list = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    predictions = []
    for decoded in decoded_list:
        if "Output:" in decoded:
            tail = decoded.split("Output:", maxsplit=1)[-1].strip()
        else:
            lines = [ln.strip() for ln in decoded.splitlines() if ln.strip()]
            tail = lines[-1] if lines else ""

        if not tail:
            predictions.append("__UNKNOWN__")
            continue

        prediction = tail.splitlines()[0].strip()

        if prediction in labels:
            predictions.append(prediction)
            continue

        matched = "__UNKNOWN__"
        for label in labels:
            if label in prediction:
                matched = label
                break
        predictions.append(matched)

    return predictions

subset_df = test_local_df.head(200)

texts = subset_df["text"].tolist()
lora_pred = batch_predict_intent(texts, label_list)
lora_acc = accuracy_score(subset_df["intent"], lora_pred)
lora_macro_f1 = f1_score(subset_df["intent"], lora_pred, average="macro")

print("LoRA accuracy:", round(lora_acc, 4))
print("LoRA macro F1:", round(lora_macro_f1, 4))

LoRA accuracy: 0.2006
LoRA macro F1: 0.2245


## 14. Compare Results

The whole point of the lab is to compare methods under the same task, split, and metric.
This final table keeps the comparison compact and explicit.


In [19]:
results_df = pd.DataFrame(
    [
        {'model': 'TF-IDF + LinearSVC', 'accuracy': svm_acc, 'macro_f1': svm_macro_f1},
        {'model': 'LFM2.5-1.2B + LoRA', 'accuracy': lora_acc, 'macro_f1': lora_macro_f1},
    ]
)

results_df


,model,accuracy,macro_f1
0,TF-IDF + LinearSVC,0.950000,0.949934
1,LFM2.5-1.2B + LoRA,0.200556,0.224475


## OOS Evaluation

The CLINC dataset also provides explicit out-of-scope examples.
A natural extension is to add the `oos` label to training and then measure OOS recall separately, following the original benchmark setup.


In [20]:
print('OOS validation examples:', len(oos_val_df))
print(oos_val_df.head())


OOS validation examples: 100
                                                text intent
0  set a warning for when my bank account starts ...    oos
1                                 a show on broadway    oos
2                 who has the best record in the nfl    oos
3                 how do i find the area of a circle    oos
4                  how many onions do i have on hand    oos


## Notes for Instructors

- The raw CLINC/OOS JSON schema may vary slightly across mirrors. The loader above is intentionally defensive, but it still should be checked once before class.
- The LoRA `target_modules` names may require one quick verification for `LiquidAI/LFM2.5-1.2B-Instruct`.
- The current inference step uses simple generated-label matching. For better stability, a label-scoring or constrained-decoding variant can be added later.
- The OOS part is intentionally left as an extension so that the core lab remains feasible inside one class session.
